# TF-IDF topics

Classic baselines on the shared 100k-complaint sample (`sample_complaints()`,
cleaned `TEXT`), built in `nhtsa.topics`:

- **NMF topics** on TF-IDF: interpretable word lists; each complaint is assigned
  its strongest topic.
- **LSA + k-means**: TF-IDF reduced to 100 dimensions, then k-means.

Both are scored with `nhtsa.evaluate.scores`: agreement with component labels
(NMI/ARI on single-label complaints, excluding `unknown`) and `era_accuracy`
(how well clusters predict 2017–20 vs 2022–25; 0.5 = no era signal).

In [1]:
import numpy as np
import polars as pl
from IPython.display import display

from nhtsa.dataset import sample_complaints
from nhtsa.evaluate import scores
from nhtsa.notebook import (
    cluster_names,
    cluster_table,
    label_share_chart,
    show_texts,
    year_share_chart,
)
from nhtsa.topics import (
    lsa_kmeans,
    nmf_topics,
    strongest_topic,
    tfidf_features,
    top_terms,
)

pl.Config.set_tbl_rows(40).set_fmt_str_lengths(120).set_float_precision(3);

In [2]:
sample = sample_complaints()
tfidf, terms = tfidf_features(sample["TEXT"].to_list())
print(
    f"{sample.height:,} complaints, received {sample['YEAR'].min()}–"
    f"{sample['YEAR'].max()}; {tfidf.shape[1]:,} TF-IDF terms (words and word pairs)."
)

100,000 complaints, received 1995–2026; 52,098 TF-IDF terms (words and word pairs).


## NMF topics

`top_label` / `label_share`: the most common component label among the topic's
single-label complaints, and its share. High share = the topic lines up with a
component; low share = a cross-cutting theme (recalls, dealer experience, …).

In [3]:
N_TOPICS = 25  # the number of component labels

doc_topics, topic_terms = nmf_topics(tfidf, N_TOPICS)
nmf_clusters = strongest_topic(doc_topics)
topic_words = top_terms(topic_terms, terms)
topic_names = cluster_names(topic_words)
cluster_table(nmf_clusters, sample["LABELS"], topic_words)

name,complaints,top_label,label_share,top_words
str,u32,str,f64,str
"""00 campaign, repair, unavailable""",1627,"""air_bags""",0.652,"""campaign, repair, unavailable, unavailable manufacturer, repair unavailable, exceeded reasonable, recall repair, manufac…"
"""01 problem, miles, told""",4082,"""engine""",0.268,"""problem, miles, told, ford, dealership, warranty, said, dealer, fix, new"""
"""02 vehicle taken, taken, diagnosed""",4856,"""engine""",0.253,"""vehicle taken, taken, diagnosed, taken dealer, needed replaced, needed, illuminated, replaced, vehicle, dealer"""
"""03 bag, air bag, air""",2038,"""air_bags""",0.862,"""bag, air bag, air, light, bag light, passenger, bag warning, passenger air, illuminated, warning light"""
"""04 steering, power steering, power""",6362,"""steering""",0.807,"""steering, power steering, power, wheel, steering wheel, turn, column, steering column, difficult, rack"""
"""05 engine, light, check""",5168,"""engine""",0.794,"""engine, light, check, oil, check engine, engine light, came, light came, coolant, engine warning"""
"""06 brake, brakes, pedal""",6497,"""brakes""",0.864,"""brake, brakes, pedal, brake pedal, abs, stop, floor, rotors, braking, applied"""
"""07 seat, belt, seat belt""",3727,"""seat_belts""",0.390,"""seat, belt, seat belt, passenger, driver seat, passenger seat, driver, buckle, child, belts"""
"""08 available, recall repair, repair available""",1380,"""air_bags""",0.260,"""available, recall repair, repair available, repair, campaign, reasonable, reasonable time, available manufacturer, recal…"


In [4]:
label_share_chart(nmf_clusters, sample["LABELS"], topic_names)

alt.Chart(...)

In [5]:
year_share_chart(nmf_clusters, sample["YEAR"], topic_names)

alt.Chart(...)

## LSA + k-means

In [6]:
kmeans_clusters = lsa_kmeans(tfidf, N_TOPICS)

## Scores

The random row is a reference: what the scores look like with no structure.

In [7]:
methods = {
    "NMF topics": nmf_clusters,
    "LSA + k-means": kmeans_clusters,
    "random (reference)": np.random.default_rng(0).integers(0, N_TOPICS, sample.height),
}
pl.DataFrame(
    [
        {"method": name, **scores(c, sample["LABELS"], sample["YEAR"].to_numpy())}
        for name, c in methods.items()
    ]
)

method,nmi,ari,coverage,era_accuracy,unassigned
str,f64,f64,f64,f64,f64
"""NMF topics""",0.350,0.228,0.740,0.594,0.000
"""LSA + k-means""",0.366,0.189,0.740,0.570,0.000
"""random (reference)""",0.001,0.000,0.740,0.511,0.000


## Read a topic

The complaints that weigh most on the chosen topic. Change `TOPIC` (the number
in the topic's name) and re-run the cell.

In [8]:
TOPIC = 0

print(topic_names[TOPIC])
for i in doc_topics[:, TOPIC].argsort()[::-1][:6]:
    r = sample.row(int(i), named=True)
    display(
        show_texts(f"{r['ODINO']} · {r['YEAR']} · {', '.join(r['LABELS'])}", r["TEXT"])
    )

00 campaign, repair, unavailable
